In [1]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

def nltk_clean_text(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', str(text), flags=re.MULTILINE)  # Convert to string
    # Remove user @ references and '#' from tweet
    text = re.sub(r'\@\w+|\#', '', str(text))  # Convert to string
    # Tokenize text
    tokens = word_tokenize(text)
    # Convert to lower case
    tokens = [w.lower() for w in tokens]
    # Remove punctuation from each word
    words = [word for word in tokens if word.isalpha()]
    # Filter out stop words
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if not w in stop_words]
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized)

def categorize_rating(rating):
    rating = int(rating)  # Convert rating to integer
    if rating > 6:
        return 'positive'
    elif rating == 6:
        return 'neutral'
    else:
        return 'negative'


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\joseph.persteins\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\joseph.persteins\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\joseph.persteins\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
# Preprocessing steps (nltk_clean_text, categorize_rating) remain the same
import pandas as pd

# Load the dataset
df = pd.read_csv(r"C:\Users\joseph.persteins\Downloads\USE THIS FOR NLP 2.csv")
# Assuming 'df' is your DataFrame containing the data
df['rating_category'] = df['rating'].apply(categorize_rating)

# Apply the improved cleaning function
df['review'] = df['review'].apply(nltk_clean_text)
df['Extracted Information'] = df['Extracted Information'].apply(nltk_clean_text)
# Assuming 'condition' column exists and needs preprocessing similar to 'review'
df['condition'] = df['condition'].apply(nltk_clean_text)
df['drug_name_x'] = df['drug_name_x'].apply(nltk_clean_text)

# Combine text columns and include 'condition'
df['combined_text'] = df['review'] + " " + df['drug_name_x'] + " " + df['rating_category']

In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
from transformers import LongformerTokenizer, TFLongformerForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [5]:
def encode_reviews_longformer(reviews, tokenizer):
    return tokenizer.batch_encode_plus(
        reviews.to_list(),
        padding='max_length',
        truncation=True,
        max_length=1024,  # Or another value suitable for your texts
        return_tensors='tf'
    )

In [6]:
from tensorflow.keras.layers import Dense  # Import the Dense layer from TensorFlow

tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096')
model = TFLongformerForSequenceClassification.from_pretrained('allenai/longformer-base-4096', num_labels=3)

from tensorflow.keras.initializers import TruncatedNormal

# Define a layer with TruncatedNormal initializer and seed
initializer = TruncatedNormal(mean=0.0, stddev=0.05, seed=42)
dense_layer = Dense(units=64, activation='relu', kernel_initializer=initializer)

from tensorflow.keras.initializers import TruncatedNormal

# Create a new instance of TruncatedNormal initializer each time
initializer1 = TruncatedNormal(mean=0.0, stddev=0.05, seed=42)
initializer2 = TruncatedNormal(mean=0.0, stddev=0.05, seed=43)

# Use initializer1 for one layer
dense_layer1 = Dense(units=64, activation='relu', kernel_initializer=initializer1)

# Use initializer2 for another layer
dense_layer2 = Dense(units=64, activation='relu', kernel_initializer=initializer2)

C:\Users\joseph.persteins\AppData\Local\anaconda31\Lib\site-packages\tf_keras\src\initializers\initializers.py:121: UserWarning: The initializer TruncatedNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(
Some layers from the model checkpoint at allenai/longformer-base-4096 were not used when initializing TFLongformerForSequenceClassification: ['lm_head']
- This IS expected if you are initializing TFLongformerForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFLongformerForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (init

In [7]:
label_encoder = LabelEncoder()
df['rating_category_encoded'] = label_encoder.fit_transform(df['rating_category'])
labels = to_categorical(df['rating_category_encoded'], num_classes=3)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(df['combined_text'], labels, test_size=0.2, random_state=42)


X_train_enc = encode_reviews_longformer(X_train, tokenizer)
X_test_enc = encode_reviews_longformer(X_test, tokenizer)

In [8]:
train_dataset = tf.data.Dataset.from_tensor_slices(({"input_ids": X_train_enc['input_ids'], "attention_mask": X_train_enc['attention_mask']}, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices(({"input_ids": X_test_enc['input_ids'], "attention_mask": X_test_enc['attention_mask']}, y_test))

In [9]:
import tensorflow as tf
print(tf.__version__)

2.16.1


In [12]:
pip install --upgrade tensorflow transformers

Note: you may need to restart the kernel to use updated packages.


In [11]:
  # Import the Adam optimizer from TensorFlow
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model 
# Define the optimizer with the desired learning rate
# Compile the model directly with the optimizer parameters
model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_dataset.shuffle(len(X_train)).batch(4), epochs=3, batch_size=4, validation_data=test_dataset.batch(4))  # Adjust batch size and epochs as needed

ValueError: Could not interpret optimizer identifier: <keras.src.optimizers.adam.Adam object at 0x000001FF368FC210>